1. For each of the following, write out Mitchell's T, E, and P — or explain why the problem is not (yet) an
ML problem: (a) flagging fraudulent credit-card transactions; (b) converting Fahrenheit to Celsius; (c)
suggesting the next word as someone types a text message.
2. In the spam example, make ham and spam more similar ( lam=np.where(is_spam, 3, 2) for suspicious words) and rerun both approaches. What happens to the gap between the hand-coded rule and the learned model, and why?
3. Classify each scenario by paradigm, justifying in one sentence: (a) predicting tomorrow's electricity demand from historical demand and weather; (b) grouping 100,000 untagged support tickets by topic; (c) a thermostat learning to minimize energy cost while keeping occupants comfortable; (d) pretraining a model to predict deleted frames of video.
4. Pick a familiar app feature (playlist recommendations, ride-time estimates, photo tagging) and write one sentence per stage describing what that team must have done — especially stages 1, 9, and 10, which are invisible in the app itself.
5. Rerun the diagnostic pipeline with random_state set to each of 0 through 9 in train_test_split , collecting the ten accuracies. Report the smallest and the largest. Given that spread, how many digits of "0.958" are you entitled to quote to someone?
6. Delete StandardScaler() from the pipeline so that LogisticRegression(max_iter=5000) sees the raw measurements, and rerun. Then lower the budget to max_iter=100 and rerun again. Using both numbers, say what the scaler is doing for the optimizer.
7. DummyClassifier takes other strategies. Score "stratified" and "uniform" alongside "most_frequent" .
8. Which scores highest, and why is that the one your model has to beat?   yeah no


Blank Question it answers Spam-filter example
Task T What should the system do? Classify emails as spam or not spam
Experience E What data does it learn from? A corpus of emails labeled by users
Performance P How do we measure success? The fraction of new emails classified correctly

**1**
T: Classify transactions as fraudulent or valid / E: A corpus of transactions labeled by people / P: Fraction of new transactions classified correctly.
T: Convert Fahrenheit to Celsius - not an ML problem, it's a basic formula.
T: Suggest the next word as someone types / E: A massive body of conversational text such as all text messages previously sent, transcriptions of natural speech, and books/articles. P: Fraction of successfully suggested next words.

1. Problem Definition
2. Data Collection
3. Explore & Clean
4. Feature Engineering
5. Model Selection
6. Training
7. Compare and Tuning
8. Final Evaluation
9. Deploy
10. Monitor & Maintain

In [10]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
rng = np.random.default_rng(42)
# Synthetic data: 400 emails. Spam tends to have more suspicious
# words and more exclamation marks, but the clouds overlap.
n = 400
is_spam = rng.integers(0, 2, size=n) # 0 = ham, 1 = spam
susp_words = rng.poisson(lam=np.where(is_spam, 3, 2)) # counts
print("Suspicious Words: ", sum(susp_words)/len(susp_words)) #average count of suspicious words
exclaims = rng.poisson(lam=np.where(is_spam, 4, 1))
X = np.column_stack([susp_words, exclaims])
y = is_spam
X_train, X_test, y_train, y_test = train_test_split(
X, y, test_size=0.25, random_state=0)
# Approach 1: a hand-written rule
def rule_based(X):
    return ((X[:, 0] >= 5) | (X[:, 1] >= 4)).astype(int)
rule_acc = (rule_based(X_test) == y_test).mean()
# Approach 2: learn the rule from examples
model = LogisticRegression().fit(X_train, y_train)
learned_acc = model.score(X_test, y_test)
print(f"Hand-coded rule accuracy: {rule_acc:.2f}")
print(f"Learned model accuracy: {learned_acc:.2f}")
print("Learned weights:", model.coef_.round(2), model.intercept_.round(2))
# Hand-coded rule accuracy: 0.91
# Learned model accuracy: 0.95
# Learned weights: [[0.91 1.04]] [-5.67]

Suspicious Words:  2.5425
Hand-coded rule accuracy: 0.82
Learned model accuracy: 0.85
Learned weights: [[0.2 1.4]] [-3.68]


When reducing the number of suspicious words, both the hand coded and learned model decreased accuracy.  

In [12]:
from sklearn.datasets import load_breast_cancer

data = load_breast_cancer()
X, y = data.data, data.target   # features, labels
print(X.shape, y.shape)
print(data.target_names)
print("malignant:", (y == 0).sum(), " benign:", (y == 1).sum())

from sklearn.model_selection import train_test_split  # noqa: I001
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline

i=0
while(i<=9):
    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=0.25, random_state=i, stratify=y)
    model = make_pipeline(StandardScaler(),
                          LogisticRegression(max_iter=5000))
    model.fit(X_tr, y_tr)      # "learning" happens here
    print(data.target_names[model.predict(X_te[:1])][0])
    print(f"accuracy on held-out data: {model.score(X_te, y_te):.3f}", "random state: ", i)  #original 0.958, random state 0
    i += 1

(569, 30) (569,)
['malignant' 'benign']
malignant: 212  benign: 357
benign
accuracy on held-out data: 0.958 random state:  0
malignant
accuracy on held-out data: 0.965 random state:  1
benign
accuracy on held-out data: 0.972 random state:  2
malignant
accuracy on held-out data: 0.986 random state:  3
malignant
accuracy on held-out data: 0.972 random state:  4
benign
accuracy on held-out data: 0.979 random state:  5
benign
accuracy on held-out data: 0.993 random state:  6
benign
accuracy on held-out data: 0.979 random state:  7
benign
accuracy on held-out data: 0.972 random state:  8
benign
accuracy on held-out data: 0.979 random state:  9


In [15]:
from sklearn.datasets import load_breast_cancer

data = load_breast_cancer()
X, y = data.data, data.target   # features, labels
print(X.shape, y.shape)
print(data.target_names)
print("malignant:", (y == 0).sum(), " benign:", (y == 1).sum())

from sklearn.model_selection import train_test_split  # noqa: I001
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline

i=0
while(i<=9):
    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=0.25, random_state=i, stratify=y)
    model = make_pipeline(LogisticRegression(max_iter=5000))
    model.fit(X_tr, y_tr)      # "learning" happens here
    print(data.target_names[model.predict(X_te[:1])][0])
    print(f"accuracy on held-out data: {model.score(X_te, y_te):.3f}", "random state: ", i)  #original 0.958, random state 0
    i += 1

(569, 30) (569,)
['malignant' 'benign']
malignant: 212  benign: 357
benign
accuracy on held-out data: 0.937 random state:  0
malignant
accuracy on held-out data: 0.944 random state:  1
benign
accuracy on held-out data: 0.958 random state:  2
malignant
accuracy on held-out data: 0.958 random state:  3
malignant
accuracy on held-out data: 0.965 random state:  4
benign
accuracy on held-out data: 0.965 random state:  5
benign
accuracy on held-out data: 0.979 random state:  6
benign
accuracy on held-out data: 0.944 random state:  7
benign
accuracy on held-out data: 0.951 random state:  8
benign
accuracy on held-out data: 0.958 random state:  9


In [28]:
from sklearn.dummy import DummyClassifier

print(f"train accuracy: {model.score(X_tr, y_tr):.3f}")
print(f"test  accuracy: {model.score(X_te, y_te):.3f}")

dumb = DummyClassifier(strategy="uniform").fit(X_tr, y_tr)  #most_frequent: 0.991, 0.958, 0.629 // stratified: 0.991, 0.958 0.490 // uniform: 0.991, 0.958, 0.476
print(f"always 'benign': {dumb.score(X_te, y_te):.3f}")

train accuracy: 0.991
test  accuracy: 0.958
always 'benign': 0.483
